# Tutorial 8 – Equity Screening & Stock Selection Case
## FINM3422 – Professional Equity Research Workflow



---

### Scenario
You are part of a junior **equity research team** at an asset management firm.

Management has asked you to:

> **Screen a universe of NASDAQ-listed technology companies and recommend ONE stock**
> for deeper fundamental analysis later in the course.

This notebook follows the **professional modelling structure** discussed in Lecture 6,
mirroring how real equity research teams generate investment ideas before valuation.

## 1. Environment & Imports 

Professional modelling always begins by making the computing environment explicit.

Why this matters:
- Reproducibility
- Team clarity
- Fewer hidden bugs

We use **yfinance**, a free and stable API wrapper around Yahoo Finance.
This is appropriate for learning, prototyping, and screening
(but not institutional trading systems).

In [ ]:
# If needed (run once): %pip install yfinance or use pip install yfinance in your terminal

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

## 2. Define the Investment Universe 

Equity research always begins with a clearly defined **investment universe**.

We restrict attention to a curated list of large-cap NASDAQ technology stocks.
Using a fixed universe ensures:
- Comparability across teams
- Stable API behaviour
- Focus on analysis, not data hunting

In [ ]:
tickers = [
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "NVDA",  # Nvidia
    "ADBE",  # Adobe
    "META",  # Meta Platforms
    "CRM",   # Salesforce
    "ORCL",  # Oracle
    "NOW",   # ServiceNow
    "INTU",  # Intuit
    "AMD"    # Advanced Micro Devices
]

## 3. Price Data Ingestion – API → DataFrame (≈15 minutes)

In professional workflows, analysts **do not manually download CSV files**.
Instead, data is pulled programmatically via APIs.

Tasks:
1. Download **monthly adjusted prices**
2. Keep adjusted close prices only
3. Ensure time is represented as a `DatetimeIndex`
4. Sort data chronologically

**Professional rule:**  
Time lives in the index. Assets live in columns.


In [ ]:
prices = yf.download(
    tickers,
    period="max",
    interval="1mo",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices.dropna(how="all").sort_index()


## 4. Data Inspection & Sanity Checks (≈10 minutes)

Before computing any metrics, analysts **inspect the raw data**.

Checks performed:
- Data types
- Missing values
- Index type
- Chronological ordering

This prevents **silent modelling errors** later.
``

Why you have NAs (this is normal, not an error)
Your table shows:

Different tickers have different listing dates
Some firms are much younger:

META, NOW → IPO mid‑2010s
CRM → 2004


Others go back decades (AAPL, MSFT)

Do not:

❌ forward‑fill prices before IPO
❌ backward‑fill fundamentals
❌ fill NAs with zeros
❌ drop rows at ingestion without thinking

In [ ]:
prices.info()

prices.isna().sum()

## 5. Returns & Performance Metrics (≈15 minutes)

Screening models often include **recent market performance** as a signal.

We compute:
- Monthly returns
- Trailing 12‑month total return

⚠️ Important:
Always fix frequency *before* computing returns.

In [ ]:
returns = prices.pct_change().dropna()

ret_12m = (1 + returns).rolling(12).apply(
    np.prod, raw=True
) - 1

latest_12m = ret_12m.iloc[-1]

## 6. Visual Diagnostics – Wealth Index (≈15 minutes)

Before trusting any metric, analysts **plot a wealth index**.

A wealth index answers:
> What happens to $1 invested over time?

This diagnostic reveals:
- Missing months
- Compounding errors
- Data misalignment

In [ ]:
(1 + returns).cumprod().plot(
    figsize=(10, 6),
    title="Wealth Index – Data Quality Diagnostic"
)
plt.show()

## 7. Fundamental Indicators – Growth & Quality (≈20 minutes)

Market prices alone are not sufficient for equity analysis.

We compute two fundamentals:
- **Revenue growth** (latest year vs previous year)
- **Operating margin** (operating income / revenue)

These act as proxies for:
- Growth
- Business quality

In [ ]:
fundamentals = []

for t in tickers:
    fin = yf.Ticker(t).financials

    rev_growth = (
        fin.loc["Total Revenue"].iloc[0] /
        fin.loc["Total Revenue"].iloc[1] - 1
    )

    op_margin = (
        fin.loc["Operating Income"].iloc[0] /
        fin.loc["Total Revenue"].iloc[0]
    )

    fundamentals.append({
        "Ticker": t,
        "RevenueGrowth": rev_growth,
        "OperatingMargin": op_margin
    })

fund_df = pd.DataFrame(fundamentals).set_index("Ticker")

## 8. Screening Table Construction (≈15 minutes)

Professional screeners combine **all signals into one master table**.

Each row represents a company.
Each column represents a screening metric.

In [ ]:
screen = fund_df.join(latest_12m.rename("Return12M"))
screen

## 9. Ranking & Shortlisting Logic (≈15 minutes)

Equity screening is about **narrowing the universe**, not precision.

Stocks are ranked on:
- Revenue growth
- Operating margin
- 12‑month performance

Ranks are combined into a simple composite score.

In [ ]:
screen["Score"] = screen.rank().mean(axis=1)

screen_sorted = screen.sort_values("Score", ascending=False)
screen_sorted

## 10. Stock Selection – Analyst Decision (≈10 minutes)

Based on the screening results, the top-ranked stock is selected
for deeper fundamental analysis later in the course.

### ✅ Selected Stock

The highest-ranked company in the composite screen is recommended
for further valuation analysis.

This mirrors how real equity research teams
move from *screening* to *valuation*.